- 避免 rl4llm 训练过程的 Reward hacking 的方式就是 cases 分析，然后在 Reward function 中进行规避
    - 核心一点就是 external Reward signal 要 faithful，policy model 正确的要打高分，错误的要打低分；
    - identify: look at your rollouts; solve: extra reward terms to penalize hack;
    - 本文的 verifier & meta-verifier 就是在不断降低 reward hacking 的过程；
- generator & verifier 是一个非常稳健高效的 workflow
    - https://gemini.google.com/app/eb54b699a18ec351
    - iterative verification & refinement 的过程就是不断提升 generation 质量的过程
    - verification rubrics 有赖于专家（对该任务的理解）的构建

### initial verifier

- $I_v$: high-level rubrics
    - 文本 prompt 驱动
- $\pi_\phi(\cdot|X,Y,I_v)$
    - first summarizes identified issues
    - assigns a score based on three levels: $s_i\in\left\{0,0.5,1\right\}$
- training
    - dataset: $\mathcal D_v=\left\{(X_i,Y_i,s_i)\right\}$
        - $Y_i$: 模型产生
        - $s_i$: 数学专家标注
    $$
    \max_{\pi_\varphi} \mathbb{E}_{(X_i, Y_i, s_i) \sim \mathcal{D}_v, (V'_i, s'_i) \sim \pi_\varphi(\cdot|X_i, Y_i)} \left[ R_{\text{format}}(V'_i) \cdot R_{\text{score}}(s'_i, s_i) \right]
    $$
    - 给定 $(X_i,Y_i)$, RL online 产生的是 $(V_i',s_i')$

### meta-verifier

- $I_{mv}$: 文本 prompt 驱动的 rubrics
- $\mathcal D_{mv}=\left\{(X_i,Y_i,V_i,ms_i)\right\}$
    - $ms_i$：数学专家对 $V_i$ 进行打分标注
- meta-verifier：$\pi_\eta(\cdot|X_i,Y_i,V_i,I_{mv})$
    - a summary of issues found in the analysis itself,
    - a quality score measuring how accurate and justified the verifier’s analysis is.
- The RL objective follows the same structure as the verifier training, with format and score rewards.

### final verifier

$$
R_v=R_{\text{format}}\cdot R_{\text{score}}\cdot R_{\text{meta}}
$$

### (Proof) Generator

$$
\max_{\pi_\theta} \mathbb{E}_{X_i \sim \mathcal{D}_p, Y_i \sim \pi_\theta(\cdot|X_i)} [R_Y]
$$
- $R_Y$: $\pi_\phi(\cdot|X_i,Y_i,I_v)$

### Self-Verification (Towards Self-Verifiable Mathematical Reasoning)

> one-shot => iterative verification & refinement 交互

- 这个学生（模型）其实有能力改正错误（can refine based on external feedback），但他缺乏自己发现错误的能力（fails to evaluate its own work），特别是当他既充当“作者”又充当“检查员”的时候，他做不到像外部专门的检查员那样严厉和客观。

$$
\begin{align*}
R &= R_{\text{format}}(Y, Z) \cdot (\alpha \cdot R_Y + \beta \cdot R_Z) \\
R_Z &= R_{\text{score}}(s', s) \cdot R_{\text{meta}}(Z)
\end{align*}
$$

- endow the proof generator with genuine verification capabilities.